# Phase 9.3 — Convert parquet files to Delta tables

Reads 10 parquet files from `Files/raw/` and writes them as Delta tables in the
`dbo` schema of `lh_travel_marketing_v2`.

In [ ]:
spark.conf.set("spark.sql.parquet.vorder.default", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.enabled", "true")
spark.conf.set("spark.microsoft.delta.optimizeWrite.binSize", "1073741824")
print("V-Order ON, optimizeWrite ON")

In [ ]:
from datetime import datetime, timezone
tables = ["customer", "hotel", "flight", "campaign", "booking",
          "payment", "cancellation", "itinerary_item", "tour_review", "inquiry"]
results = []
for tbl in tables:
    src = f"Files/raw/{tbl}.parquet"
    print(f"[{datetime.now(timezone.utc).isoformat()}] Loading {src} -> dbo.{tbl}")
    df = spark.read.parquet(src)
    df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable(f"dbo.{tbl}")
    cnt = spark.table(f"dbo.{tbl}").count()
    results.append((tbl, cnt, len(df.columns)))
    print(f"  -> {cnt:,} rows, {len(df.columns)} cols")
print("\n=== LOAD SUMMARY ===")
for tbl, cnt, cols in results:
    print(f"  dbo.{tbl:20s} rows={cnt:>8,d}  cols={cols}")

In [ ]:
tables = ["customer", "hotel", "flight", "campaign", "booking",
          "payment", "cancellation", "itinerary_item", "tour_review", "inquiry"]
for tbl in tables:
    print(f"OPTIMIZE dbo.{tbl} VORDER ...")
    try:
        spark.sql(f"OPTIMIZE dbo.{tbl} VORDER").show(truncate=False)
    except Exception as ex:
        print(f"  VORDER variant failed ({ex}); retrying plain OPTIMIZE")
        spark.sql(f"OPTIMIZE dbo.{tbl}").show(truncate=False)
print("OPTIMIZE complete")

In [ ]:
expected = {
    "customer": 10000, "hotel": 500, "flight": 2000, "campaign": 200,
    "booking": 50000, "payment": 61289, "cancellation": 5135,
    "itinerary_item": 175323, "tour_review": 8243, "inquiry": 20000,
}
print(f"{'table':22s} {'actual':>10s}  {'expected':>10s}  match")
all_ok = True
for tbl, exp in expected.items():
    actual = spark.table(f"dbo.{tbl}").count()
    ok = actual == exp
    all_ok = all_ok and ok
    flag = "OK" if ok else "MISMATCH"
    print(f"  dbo.{tbl:18s} {actual:>10,d}  {exp:>10,d}  {flag}")
print(f"\nAll counts match: {all_ok}")

In [ ]:
fk_checks = [
    ("payment -> booking",
     "SELECT COUNT(*) AS orphans FROM dbo.payment p LEFT JOIN dbo.booking b ON p.booking_id = b.booking_id WHERE b.booking_id IS NULL"),
    ("tour_review -> booking",
     "SELECT COUNT(*) AS orphans FROM dbo.tour_review r LEFT JOIN dbo.booking b ON r.booking_id = b.booking_id WHERE b.booking_id IS NULL"),
    ("booking -> customer",
     "SELECT COUNT(*) AS orphans FROM dbo.booking b LEFT JOIN dbo.customer c ON b.customer_id = c.customer_id WHERE c.customer_id IS NULL"),
    ("cancellation -> booking",
     "SELECT COUNT(*) AS orphans FROM dbo.cancellation x LEFT JOIN dbo.booking b ON x.booking_id = b.booking_id WHERE b.booking_id IS NULL"),
    ("itinerary_item -> booking",
     "SELECT COUNT(*) AS orphans FROM dbo.itinerary_item i LEFT JOIN dbo.booking b ON i.booking_id = b.booking_id WHERE b.booking_id IS NULL"),
]
for label, q in fk_checks:
    n = spark.sql(q).first()["orphans"]
    flag = "OK" if n == 0 else "FAIL"
    print(f"  {label:30s} orphans={n}  {flag}")

In [ ]:
print("=== Bookings by year ===")
spark.sql("""
    SELECT YEAR(departure_date) AS yr, COUNT(*) AS bookings, SUM(total_revenue_jpy) AS revenue_jpy
    FROM dbo.booking GROUP BY YEAR(departure_date) ORDER BY yr
""").show(truncate=False)

print("=== Bookings by destination_type ===")
spark.sql("""
    SELECT destination_type, COUNT(*) AS bookings,
           ROUND(AVG(price_per_person_jpy), 0) AS avg_price_jpy,
           ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM dbo.booking), 2) AS pct
    FROM dbo.booking GROUP BY destination_type ORDER BY bookings DESC
""").show(truncate=False)

In [ ]:
print("=== Tables in dbo schema ===")
spark.sql("SHOW TABLES IN dbo").show(truncate=False)